## Library Imports

In [1]:
import pandas as pd

## Directory Setup

In [ ]:
# Define input directory
input_dir = ('')
# Define output directory
output_dir = ('')

## Geography Spine Loading

In [3]:
# Load Geography Spine
itl2_spine = pd.read_csv('geography_spine.csv')
itl2_spine

,la_code,itl2_code,itl2_name
0,E06000001,TLC3,Tees Valley
1,E06000002,TLC3,Tees Valley
2,E06000003,TLC3,Tees Valley
3,E06000004,TLC3,Tees Valley
4,E06000005,TLC3,Tees Valley
...,...,...,...
357,N09000007,TLN0,Northern Ireland
358,N09000008,TLN0,Northern Ireland
359,N09000009,TLN0,Northern Ireland
360,N09000010,TLN0,Northern Ireland


## Housing Price

### Data Loading

In [4]:
# Load Housing Price csv file
housing_price_df = pd.read_csv(f'{input_dir}/average-house-prices-2026-01.csv')
housing_price_df

,Date,Region_Name,Area_Code,Average_Price,Monthly_Change,Annual_Change,Average_Price_SA
0,1968-04-01,Northern Ireland,N92000002,3465,NaN,NaN,NaN
1,1968-04-01,England,E92000001,3218,NaN,NaN,NaN
2,1968-04-01,Wales,W92000004,2732,NaN,NaN,NaN
3,1968-04-01,Scotland,S92000003,2738,NaN,NaN,NaN
4,1968-04-01,London,E12000007,4730,NaN,NaN,NaN
...,...,...,...,...,...,...,...
149485,2026-01-01,Caerphilly,W06000018,193058,-1.0,1.2,NaN
149486,2026-01-01,Blaenau Gwent,W06000019,139073,-3.0,4.5,NaN
149487,2026-01-01,England and Wales,K04000001,285111,-0.3,1.2,285203.0
149488,2026-01-01,Great Britain,K03000001,270707,-0.3,1.2,270741.0


### Data Transformation

In [5]:
# Extract year from date
housing_price_df['year'] = pd.to_datetime(housing_price_df['Date']).dt.year

# Keep only local authority district level data (1995 onwards)
housing_price_df = housing_price_df[housing_price_df['year'] >= 1995]

# Map LA codes to ITL2
housing_price_df = housing_price_df.merge(itl2_spine, left_on='Area_Code', right_on='la_code', how='left')

# Check for unmatched local authorities
unmatched = housing_price_df[housing_price_df['itl2_code'].isna()]
if len(unmatched):
    print(f"{unmatched['la_code'].nunique()} LA codes did not match:")
    print(unmatched['la_code'].unique())

# Collapse monthly to yearly, grouped by ITL2
housing_price_panel = (
    housing_price_df.groupby(['itl2_code', 'itl2_name', 'year'])['Average_Price']
    .mean()
    .reset_index()
    .rename(columns={'Average_Price': 'average_house_price'})
)

# Sort by year, then ITL2 code
housing_price_panel = housing_price_panel.sort_values(['year', 'itl2_code']).reset_index(drop=True)
housing_price_panel

0 LA codes did not match:
[nan]


,itl2_code,itl2_name,year,average_house_price
0,TLC3,Tees Valley,1995,40850.250000
1,TLC4,"Northumberland, Durham and Tyne & Wear",1995,40535.845238
2,TLD1,Cumbria,1995,43739.458333
3,TLD3,Greater Manchester,1995,40652.991667
4,TLD4,Lancashire,1995,43579.113095
...,...,...,...,...
1403,TLM2,Highlands and Islands,2026,177217.142857
1404,TLM3,West Central Scotland,2026,186517.285714
1405,TLM5,North Eastern Scotland,2026,165793.500000
1406,TLM9,Southern Scotland,2026,160530.833333


## Median Affordability Ratio - Work-place based
Ratio of median house price to median gross annual workplace-based earnings

### Data Loading

In [6]:
# Load Affordability Ratio excel file
afford_work_df = pd.read_excel(f'{input_dir}/aff1ratioofhousepricetoworkplacebasedearnings.xlsx', sheet_name='5c', skiprows=1)
afford_work_df

,Country/Region code,Country/Region name,Local authority code,Local authority name,1997,1998,1999,2000,2001,2002,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,5-Year Average
0,E12000001,North East,E06000001,Hartlepool,2.64,2.74,2.85,3.02,2.75,2.9,...,4.8,4.82,5.00,4.63,4.94,4.65,4.73,4.59,4.8,4.74
1,E12000001,North East,E06000002,Middlesbrough,2.83,2.74,2.89,2.73,2.58,2.47,...,4.88,5.28,5.15,5,5.42,4.95,4.74,4.8,4.51,4.88
2,E12000001,North East,E06000003,Redcar and Cleveland,2.34,2.45,2.12,2.21,2.39,2.69,...,5.11,5.11,5.03,5.15,5.84,5.67,5.2,5.38,4.68,5.35
3,E12000001,North East,E06000004,Stockton-on-Tees,2.97,2.93,3.05,3.06,3.07,3.53,...,5.64,5.5,5.06,5.01,5.56,5.22,5.73,5.58,4.78,5.37
4,E12000001,North East,E06000005,Darlington,3.04,3.46,3.14,3.24,3.19,3.18,...,5.37,5.34,5.04,5.35,5.53,5.27,5.45,4.8,5.26,5.26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,W92000004,Wales,W06000020,Torfaen,2.46,2.63,2.69,2.85,2.72,3.05,...,4.85,5.31,5.41,5.49,5.82,5.59,5.42,5.02,5.55,5.48
314,W92000004,Wales,W06000021,Monmouthshire,4.15,4.72,4.64,5.31,5.97,6.97,...,8.42,8.66,8.69,8.59,10.25,9.96,9.41,8.57,7.98,9.23
315,W92000004,Wales,W06000022,Newport,2.87,2.67,2.9,3.15,3.25,3.77,...,5.99,6.08,6.18,6.31,7.35,6.9,7.46,6.88,6.28,6.97
316,W92000004,Wales,W06000023,Powys,3.68,3.71,3.91,4.38,4.03,4.66,...,7.46,7.51,6.90,6.31,8.02,8.21,7.91,7.34,7.86,7.87


### Data Transformation

In [7]:
# Replace "[x]" placeholders with nulls
afford_work_df = afford_work_df.replace('[x]', pd.NA)

# Map LA codes to ITL2
afford_work_df = afford_work_df.merge(itl2_spine, left_on='Local authority code', right_on='la_code', how='left')

# Check for unmatched local authorities
unmatched = afford_work_df[afford_work_df['itl2_code'].isna()]
if len(unmatched):
    print(f"{unmatched['la_code'].nunique()} LA codes did not match:")
    print(unmatched['la_code'].unique())

# Reshape wide to long format
# Drop unnecessary columns
afford_work_df.drop(columns=['Country/Region code', 'Country/Region name', 'Local authority name', 'Local authority code', '5-Year Average', 'la_code'], inplace=True)
# Define ID vars to stay fixed
id_vars = ['itl2_code', 'itl2_name']
year_cols = [c for c in afford_work_df.columns if c not in id_vars]
afford_work_panel = afford_work_df.melt(id_vars=id_vars, value_vars=year_cols, var_name='year', value_name='affordability_ratio_workplace')

# Collapse multiple LAs into one ITL2-level average per year
afford_work_panel = (afford_work_panel.groupby(['itl2_code', 'itl2_name', 'year'])['affordability_ratio_workplace'].mean().reset_index())

# Sort by year, then ITL2 code
afford_work_panel = afford_work_panel.sort_values(['year', 'itl2_code']).reset_index(drop=True)
afford_work_panel

,itl2_code,itl2_name,year,affordability_ratio_workplace
0,TLC3,Tees Valley,1997,2.764
1,TLC4,"Northumberland, Durham and Tyne & Wear",1997,3.1
2,TLD1,Cumbria,1997,NaN
3,TLD3,Greater Manchester,1997,2.9
4,TLD4,Lancashire,1997,2.876429
...,...,...,...,...
1126,TLK6,"North Somerset, Somerset and Dorset",2025,9.0225
1127,TLK7,Gloucestershire and Wiltshire,2025,8.6575
1128,TLL3,North Wales,2025,5.895
1129,TLL4,Mid and South West Wales,2025,6.09


## Median Affordability Ratio - Residence based
Ratio of median house price to median gross annual residence-based earnings

### Data Loading

In [8]:
# Load Affordability Ratio excel file
afford_res_df = pd.read_excel(f'{input_dir}/aff2ratioofhousepricetoresidencebasedearnings.xlsx', sheet_name='5c', skiprows=1)
afford_res_df

,Country/Region code,Country/Region name,Local authority code,Local authority name,2002,2003,2004,2005,2006,2007,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,5-Year Average
0,E12000001,North East,E06000001,Hartlepool,2.86,3.02,2.82,2.86,4.14,4.63,...,4.82,4.6,4.9,4.54,5.05,4.45,4.8,4.78,4.86,4.79
1,E12000001,North East,E06000002,Middlesbrough,2.83,2.65,3.74,4.23,5.17,5.4,...,5.18,5.23,5.18,5.07,5.33,5.1,5.16,4.73,4.36,4.94
2,E12000001,North East,E06000003,Redcar and Cleveland,2.83,3.95,4.37,4.67,5.66,5.52,...,5.12,5.36,5.03,5.06,5.84,5.12,5.24,5.39,4.7,5.26
3,E12000001,North East,E06000004,Stockton-on-Tees,3.33,4.03,4.54,4.99,5.53,5.64,...,5.37,5.43,4.99,5.19,5.67,5.29,5.35,5.1,4.66,5.21
4,E12000001,North East,E06000005,Darlington,3.42,4.06,5.12,5.96,5.94,5.45,...,4.96,5.26,5.07,5.55,5.32,5.49,5.42,4.94,4.92,5.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,W92000004,Wales,W06000020,Torfaen,2.95,3.63,4.31,4.44,4.79,5.04,...,4.68,5.28,5.5,5.56,6.51,5.88,6,5.7,5.11,5.84
314,W92000004,Wales,W06000021,Monmouthshire,5.43,5.68,6.88,7.42,7.29,7.33,...,7.16,7.28,7.27,7.56,8.53,8.12,9.05,8.44,7.46,8.32
315,W92000004,Wales,W06000022,Newport,3.76,4.22,5.3,5.57,5.85,6.17,...,6,5.91,6.18,6.03,7.09,6.57,6.91,6.27,6.35,6.64
316,W92000004,Wales,W06000023,Powys,4.57,5.18,7.26,7.37,7.76,7.9,...,6.99,6.89,6.68,6.35,7.96,8.33,7.1,6.84,6.82,7.41


### Data Transformation

In [9]:
# Replace "[x]" placeholders with nulls
afford_res_df = afford_res_df.replace('[x]', pd.NA)

# Map LA codes to ITL2
afford_res_df = afford_res_df.merge(itl2_spine, left_on='Local authority code', right_on='la_code', how='left')

# Check for unmatched local authorities
unmatched = afford_res_df[afford_res_df['itl2_code'].isna()]
if len(unmatched):
    print(f"{unmatched['la_code'].nunique()} LA codes did not match:")
    print(unmatched['la_code'].unique())

# Reshape wide to long format
# Drop unnecessary columns
afford_res_df.drop(columns=['Country/Region code', 'Country/Region name', 'Local authority name', 'Local authority code', '5-Year Average', 'la_code'], inplace=True)
# Define ID vars to stay fixed
id_vars = ['itl2_code', 'itl2_name']
year_cols = [c for c in afford_res_df.columns if c not in id_vars]
afford_res_panel = afford_res_df.melt(id_vars=id_vars, value_vars=year_cols, var_name='year', value_name='affordability_ratio_residence')

# Collapse multiple LAs into one ITL2-level average per year
afford_res_panel = (afford_res_panel.groupby(['itl2_code', 'itl2_name', 'year'])['affordability_ratio_residence'].mean().reset_index())

# Sort by year, then ITL2 code
afford_res_panel = afford_res_panel.sort_values(['year', 'itl2_code']).reset_index(drop=True)
afford_res_panel

,itl2_code,itl2_name,year,affordability_ratio_residence
0,TLC3,Tees Valley,2002,3.054
1,TLC4,"Northumberland, Durham and Tyne & Wear",2002,3.354
2,TLD1,Cumbria,2002,NaN
3,TLD3,Greater Manchester,2002,3.422
4,TLD4,Lancashire,2002,3.165
...,...,...,...,...
931,TLK6,"North Somerset, Somerset and Dorset",2025,8.5325
932,TLK7,Gloucestershire and Wiltshire,2025,8.12125
933,TLL3,North Wales,2025,5.915
934,TLL4,Mid and South West Wales,2025,5.795


## Output Saving

In [10]:
# Define dataframes to save
dfs_to_save = {
    'house_prices.csv': housing_price_panel,
    'housing_affordability_ratio_workplace': afford_work_panel,
    'housing_affordability_ratio_residence': afford_res_panel
}

# Save files
for filename, df in dfs_to_save.items():
    df.to_csv(f'{output_dir}/{filename}', index=False)